DETECÇÃO DE LADDER LOGIC BOMB

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from pycaret.classification import (
    setup,
    compare_models,
    pull,
    finalize_model,
    predict_model,
    save_model,
    create_model,
    plot_model,
)



In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
DATA_PATH = os.getcwd()
normal_file = os.path.join(DATA_PATH, "dataset_normal.csv")
malicioso_file = os.path.join(DATA_PATH, "dataset_malicious.csv")
df_normal = pd.read_csv(normal_file)
df_malicioso = pd.read_csv(malicioso_file)

In [4]:
df_normal.shape

(90, 31)

In [5]:
df_normal = pd.read_csv(normal_file)
df_malicioso = pd.read_csv(malicioso_file)

# corrigir orientação
df_normal = df_normal.set_index(df_normal.columns[0]).T.reset_index()
df_malicioso = df_malicioso.set_index(df_malicioso.columns[0]).T.reset_index()

# renomear a coluna do índice
df_normal = df_normal.rename(columns={"index": "arquivo_xml"})
df_malicioso = df_malicioso.rename(columns={"index": "arquivo_xml"})

df_normal.columns.name = None
df_malicioso.columns.name = None

df_normal["label"] = "normal"
df_malicioso["label"] = "malicioso"

In [6]:
df = pd.concat([df_normal, df_malicioso], ignore_index=True)
df.head()


,arquivo_xml,assigned_vars_st,avg_degree,block_add_count,block_ctd_count,block_ctu_count,block_div_count,block_eq_count,block_ge_count,block_gt_count,...,seq_has_ldn,seq_has_out,seq_n_lines,seq_n_unique_opcodes,seq_n_unique_operands,seq_opcode_vocab,st_text_length,task_intervals,task_priorities,written_output_names
0,lassignment.xml,OUT_MV1;OUT_MV2;real_value,2.1111,0,0,0,0,0,0,0,...,1,1,27,8,17,ARG:5;CALL:2;IN:3;LD:3;LDN:3;OUT:1;OUTVAR:4;RET:6,248,T#20ms,0,CYCLE_ON;MV1;MV2
1,lassignment1.xml,OUT_MV1;OUT_MV2;i;real_value,2.1429,0,0,0,0,0,0,0,...,1,1,20,8,16,ARG:5;CALL:1;IN:4;LD:3;LDN:1;OUT:1;OUTVAR:2;RET:3,319,T#20ms,0,CYCLE_ON;MV1;MV2
2,lexit.xml,OUT_MV1;OUT_MV2;i;real_value,2.1667,0,0,0,0,1,0,0,...,1,1,42,8,22,ARG:9;CALL:4;IN:5;LD:3;LDN:3;OUT:1;OUTVAR:6;RE...,438,T#20ms,0,CYCLE_ON;MV1;MV2
3,lstart_cycle.xml,OUT;OUT_MV1;OUT_MV2;real_value,2.2222,0,0,0,0,0,0,0,...,1,1,31,8,19,ARG:7;CALL:3;IN:4;LD:2;LDN:2;OUT:1;OUTVAR:4;RET:8,273,T#20ms,0,CYCLE_ON;MV1;MV2
4,lstart_cycle1.xml,OUT;OUT_MV1;OUT_MV2;real_value,2.1333,0,0,0,0,0,0,0,...,1,1,25,8,18,ARG:7;CALL:2;IN:5;LD:2;LDN:1;OUT:1;OUTVAR:2;RET:5,290,T#20ms,0,CYCLE_ON;MV1;MV2


In [7]:
print(df.shape)

(60, 91)


In [ ]:
# =========================================================
# LIMPEZA MÍNIMA
# Mantendo quase todas as features para análise de importância
# Removemos só a sequência crua, se existir
# =========================================================
DROP_COLS = [
    "il_like_sequence",
    "assigned_vars_st",
    "block_types",
    "function_block_names",
    "input_var_names",
    "local_var_names",
    "output_var_names",
    "pou_names",
    "seq_opcode_vocab",
    "written_output_names",
    "task_intervals",
    "task_priorities",
]

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

if "label" not in df.columns:
    raise ValueError("A coluna 'label' não foi encontrada.")

# converte o que puder para numérico
for col in df.columns:
    if col not in ["arquivo_xml", "label"]:
        df[col] = pd.to_numeric(df[col], errors="ignore")

# arquivo_xml fica no DF, mas não entra no treino
ignore_features = [c for c in ["arquivo_xml"] if c in df.columns]

print("\nShape final:", df.shape)
print("\nDistribuição das classes:")
print(df["label"].value_counts())


# =========================================================
# SETUP PYCARET
# - split 80/20
# - 5-fold CV
# =========================================================
exp = setup(
    data=df,
    target="label",
    ignore_features=ignore_features,
    train_size=0.8,
    fold=5,
    fold_strategy="stratifiedkfold",
    session_id=42,
    normalize=True,
    html=False,
    verbose=True,
)

# =========================================================
# COMPARAÇÃO DE MODELOS
# modelos selecionados manualmente
# =========================================================
best_model = compare_models(
    include=[
        "lr",       # Logistic Regression
        "rf",       # Random Forest
        "et",       # Extra Trees
        "mlp",      # MLP Classifier
        "gbc",      # Gradient Boosting Classifier 
        "lightgbm", # LightGBM
        "svm",      # SVM
    ]
)

leaderboard = pull()
leaderboard.to_csv("classification_leaderboard.csv", index=False)
print("\nLeaderboard salvo em classification_leaderboard.csv")

# =========================================================
# MODELO FINAL
# treina no conjunto de treino inteiro
# =========================================================
final_model = finalize_model(best_model)

# avaliação no hold-out test set (20%)
holdout_pred = predict_model(final_model)
holdout_pred.to_csv("holdout_predictions.csv", index=False)
print("Predições hold-out salvas em holdout_predictions.csv")

# predição no dataset inteiro, se quiser inspecionar
all_pred = predict_model(final_model, data=df)
all_pred.to_csv("all_predictions.csv", index=False)
print("Predições completas salvas em all_predictions.csv")

save_model(final_model, "plc_pycaret_best_model")
print("Modelo salvo em plc_pycaret_best_model.pkl")

# =========================================================
# IMPORTÂNCIA DAS FEATURES
# Usa RF para garantir interpretabilidade
# =========================================================
rf_model = create_model("rf")
rf_metrics = pull()
rf_metrics.to_csv("random_forest_metrics.csv", index=False)

try:
    plot_model(rf_model, plot="feature", save=True)
    print("Plot de importância das features salvo pelo PyCaret.")
except Exception as e:
    print("Não foi possível gerar o plot de importância:", e)

save_model(rf_model, "plc_pycaret_rf_model")
print("Modelo RF salvo em plc_pycaret_rf_model.pkl")

print("\nArquivos gerados:")
print("- classification_leaderboard.csv")
print("- holdout_predictions.csv")
print("- all_predictions.csv")
print("- plc_pycaret_best_model.pkl")
print("- random_forest_metrics.csv")
print("- plc_pycaret_rf_model.pkl")


Shape final: (60, 80)

Distribuição das classes:
label
normal       30
malicioso    30
Name: count, dtype: int64
                    Description                    Value
0                    Session id                       42
1                        Target                    label
2                   Target type                   Binary
3                Target mapping  malicioso: 0, normal: 1
4           Original data shape                 (60, 80)
5        Transformed data shape                 (60, 79)
6   Transformed train set shape                 (48, 79)
7    Transformed test set shape                 (12, 79)
8               Ignore features                        1
9              Numeric features                       78
10                   Preprocess                     True
11              Imputation type                   simple
12           Numeric imputation                     mean
13       Categorical imputation                     mode
14                    Normalize

                                    Model  Accuracy    AUC  Recall   Prec.  \
et                 Extra Trees Classifier    0.9378  0.943  0.9378  0.9481   
lr                    Logistic Regression    0.9178  0.960  0.9178  0.9315   
rf               Random Forest Classifier    0.9178  0.992  0.9178  0.9362   
mlp                        MLP Classifier    0.9178  0.976  0.9178  0.9315   
gbc          Gradient Boosting Classifier    0.9178  0.911  0.9178  0.9362   
svm                   SVM - Linear Kernel    0.8978  0.952  0.8978  0.9196   
lightgbm  Light Gradient Boosting Machine    0.4778  0.500  0.4778  0.2290   

              F1   Kappa     MCC  TT (Sec)  
et        0.9368  0.8738  0.8847     0.422  
lr        0.9166  0.8338  0.8480     0.476  
rf        0.9153  0.8338  0.8523     0.426  
mlp       0.9166  0.8338  0.8480     0.408  
gbc       0.9153  0.8338  0.8523     0.342  
svm       0.8951  0.7938  0.8156     0.010  
lightgbm  0.3094  0.0000  0.0000     0.016  

Leaderboard sa

      Accuracy    AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                         
0       0.9000  1.000  0.9000  0.9167  0.8990  0.8000  0.8165
1       0.8000  0.960  0.8000  0.8571  0.7917  0.6000  0.6547
2       1.0000  1.000  1.0000  1.0000  1.0000  1.0000  1.0000
3       0.8889  1.000  0.8889  0.9074  0.8860  0.7692  0.7906
4       1.0000  1.000  1.0000  1.0000  1.0000  1.0000  1.0000
Mean    0.9178  0.992  0.9178  0.9362  0.9153  0.8338  0.8523
Std     0.0756  0.016  0.0756  0.0559  0.0784  0.1518  0.1325
Plot de importância das features salvo pelo PyCaret.
Transformation Pipeline and Model Successfully Saved
Modelo RF salvo em plc_pycaret_rf_model.pkl

Arquivos gerados:
- classification_leaderboard.csv
- holdout_predictions.csv
- all_predictions.csv
- plc_pycaret_best_model.pkl
- random_forest_metrics.csv
- plc_pycaret_rf_model.pkl


In [159]:
from pycaret.classification import models
print(models())

                                     Name  \
ID                                          
lr                    Logistic Regression   
knn                K Neighbors Classifier   
nb                            Naive Bayes   
dt               Decision Tree Classifier   
svm                   SVM - Linear Kernel   
rbfsvm                SVM - Radial Kernel   
gpc           Gaussian Process Classifier   
mlp                        MLP Classifier   
ridge                    Ridge Classifier   
rf               Random Forest Classifier   
qda       Quadratic Discriminant Analysis   
ada                  Ada Boost Classifier   
gbc          Gradient Boosting Classifier   
lda          Linear Discriminant Analysis   
et                 Extra Trees Classifier   
lightgbm  Light Gradient Boosting Machine   
dummy                    Dummy Classifier   

                                                  Reference  Turbo  
ID                                                                  
lr    

,arquivo_xml,assigned_vars_st,avg_degree,block_add_count,block_ctd_count,block_ctu_count,block_div_count,block_eq_count,block_ge_count,block_gt_count,...,seq_has_ldn,seq_has_out,seq_n_lines,seq_n_unique_opcodes,seq_n_unique_operands,seq_opcode_vocab,st_text_length,task_intervals,task_priorities,written_output_names
0,lassignment.xml,OUT_MV1;OUT_MV2;real_value,2.1111,0,0,0,0,0,0,0,...,1,1,27,8,17,ARG:5;CALL:2;IN:3;LD:3;LDN:3;OUT:1;OUTVAR:4;RET:6,248,T#20ms,0,CYCLE_ON;MV1;MV2
1,lassignment1.xml,OUT_MV1;OUT_MV2;i;real_value,2.1429,0,0,0,0,0,0,0,...,1,1,20,8,16,ARG:5;CALL:1;IN:4;LD:3;LDN:1;OUT:1;OUTVAR:2;RET:3,319,T#20ms,0,CYCLE_ON;MV1;MV2
2,lexit.xml,OUT_MV1;OUT_MV2;i;real_value,2.1667,0,0,0,0,1,0,0,...,1,1,42,8,22,ARG:9;CALL:4;IN:5;LD:3;LDN:3;OUT:1;OUTVAR:6;RE...,438,T#20ms,0,CYCLE_ON;MV1;MV2
3,lstart_cycle.xml,OUT;OUT_MV1;OUT_MV2;real_value,2.2222,0,0,0,0,0,0,0,...,1,1,31,8,19,ARG:7;CALL:3;IN:4;LD:2;LDN:2;OUT:1;OUTVAR:4;RET:8,273,T#20ms,0,CYCLE_ON;MV1;MV2
4,lstart_cycle1.xml,OUT;OUT_MV1;OUT_MV2;real_value,2.1333,0,0,0,0,0,0,0,...,1,1,25,8,18,ARG:7;CALL:2;IN:5;LD:2;LDN:1;OUT:1;OUTVAR:2;RET:5,290,T#20ms,0,CYCLE_ON;MV1;MV2


(60, 91)